# AgentState 状态管理

Agent 在执行过程中需要维护状态——消息历史、结构化响应、流程控制等。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

## AgentState 结构

| 字段 | 类型 | 必填 | 说明 |
| --- | --- | --- | --- |
| messages | list[AnyMessage] | 是 | 消息历史，add_messages reducer |
| jump_to | str | 否 | 流程跳转（tools/model/end），ephemeral |
| structured_response | Any | 否 | 结构化输出结果 |

## messages——消息历史的 Reducer 机制

messages 字段使用 `add_messages` reducer，更新时追加而非覆盖。

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages

messages = [
    HumanMessage(content="你好", id="1"),
    AIMessage(content="你好！", id="2"),
]
new_msg = AIMessage(content="有什么可以帮你的？", id="3")
result = add_messages(messages, [new_msg])

print(f"合并前: {len(messages)} 条")
print(f"合并后: {len(result)} 条")
for msg in result:
    print(f"  [{msg.type}] {msg.content}")

`add_messages` 智能特性：同名覆盖（ID 相同替换）、支持 RemoveMessage、类型安全。

## jump_to——流程跳转控制

Middleware 中最常用的字段，ephemeral 瞬态，使用后自动清除。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

@before_model
def check_question(state, runtime):
    messages = state.get("messages", [])
    if not messages:
        return None
    if "密码" in str(messages[-1].content):
        return {"jump_to": "end", "messages": [HumanMessage(content="抱歉，不能回答关于密码的问题。")]}
    return None

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, middleware=[check_question], system_prompt="你是菜鸟教程 RUNOOB 的助手。")

print("jump_to 中间件已注册")

| jump_to 值 | 跳转到 | 效果 |
| --- | --- | --- |
| "tools" | 工具执行节点 | 跳过模型调用 |
| "model" | 模型节点 | 让模型重新处理 |
| "end" | 结束 Agent 循环 | 跳转到结束 |

## structured_response——结构化输出

使用 `response_format` 参数时，结果存储在此字段。

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

class CourseRecommendation(BaseModel):
    course_name: str = Field(description="推荐课程")
    reason: str = Field(description="推荐理由")
    difficulty: str = Field(description="难度")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, response_format=CourseRecommendation, system_prompt="你是菜鸟教程 RUNOOB 的学习顾问。")

print("结构化输出 Agent 已创建")

## 自定义 State 扩展

继承 `AgentState` 添加业务字段。

In [ ]:
from typing import Annotated
from langchain.agents import create_agent, AgentState
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool, InjectedState

class ShoppingAgentState(AgentState):
    cart: list[str]
    total_price: float

@tool
def add_to_cart(item: str, price: float, state: Annotated[dict, InjectedState]) -> str:
    cart = state.get("cart", [])
    total = state.get("total_price", 0.0)
    state["cart"] = cart + [item]
    state["total_price"] = total + price
    return f"已添加 {item} (¥{price})"

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, tools=[add_to_cart], state_schema=ShoppingAgentState, system_prompt="你是购物助手。")

print("自定义状态 Agent 已创建")

## state_schema vs middleware state_schema

| 方式 | 优先级 |
| --- | --- |
| create_agent(state_schema=...) | 全局，最高 |
| AgentMiddleware(state_schema=...) | middleware 局部，较低 |

**术语：** AgentState、add_messages、jump_to、structured_response、state_schema